# Tags Pipeline v4 — Resolution notes → Tags, Tag 1, Tag 2

Alur: **baca input (Excel/CSV)** → **ambil tags (semua blok, unik)** → **logic certificate** → **buang IS High Risk** → **Tag 1 & Tag 2 (prioritas `#`)** → **simpan Excel**.

Semua pencocokan **tidak peduli huruf besar/kecil**.

**Ambil tags** (sama seperti v3)
- Kalau ada beberapa `Tags:`, **semua** tag dari semua blok diambil lalu dibuat **unik** (urutan kemunculan pertama).
- Tag yang sama muncul dengan dan tanpa `#` (mis. `Infra` dan `#Infra`) → dihitung sebagai tag ber-`#`.
- `Tags Extracted` berisi **semua** tag (tanpa tanda `#`). `Tags Raw` berisi tag yang sama lengkap dengan `#`-nya.

**Yang baru di v4 — `#` jadi PRIORITAS untuk Tag 1 & Tag 2, bukan filter**

Di v3 tag tanpa `#` dibuang. Di v4 tag tanpa `#` tetap ada, hanya jadi cadangan:

| Kondisi | Tag 1 & Tag 2 diambil dari |
|---|---|
| Semua tag pakai `#` | semua tag, logic lama |
| Semua tag tanpa `#` | semua tag, logic lama |
| Ada **≥ 2** tag ber-`#` (+ ada yang tanpa `#`) | **hanya tag ber-`#`**, logic lama |
| Hanya **1** tag ber-`#` (+ ada yang tanpa `#`) | tag ber-`#` **wajib** masuk; slot satunya diisi dari tag tanpa `#` |

Kasus **1 tag ber-`#`** — slotnya ikut prefix-nya, slot lain diisi dari tag tanpa `#` dengan logic lama:
- Tag ber-`#` tipe **Tag 1** (`ticket_duplicated` / `cm_bug` / `cm_`) → Tag 1 = tag ber-`#`; Tag 2 = prefix Tag 2 terkuat dari tag tanpa `#`, kalau tidak ada → tag tanpa `#` paling awal.
- Tag ber-`#` tipe **Tag 2** (`attachment` / `on-call` / `FEEDBACK_COMPLIMENT` / `user_request` / `user_`) → Tag 2 = tag ber-`#`; Tag 1 = prefix Tag 1 terkuat dari tag tanpa `#` → prefix Tag 2 terkuat → tag tanpa `#` paling awal.
- Tag ber-`#` **tanpa prefix** → kalau tag tanpa `#` punya prefix Tag 1: Tag 1 = itu, Tag 2 = tag ber-`#`. Kalau punya prefix Tag 2: Tag 1 = tag ber-`#`, Tag 2 = itu. Kalau tidak ada prefix: Tag 1 = tag ber-`#`, Tag 2 = tag tanpa `#` paling awal.

Kolom `Tag 1/2 Mode` menunjukkan baris mana yang memakai aturan yang mana. Prioritas `#` bisa dimatikan lewat `HASH_PRIORITY = False`.

**Logic certificate** (dicek pada **semua** tag, ber-`#` maupun tidak)
1. Ada tag yang isinya **persis** `Certificate` (atau `Certificates`) → tags = `Certificate`. Tag dengan awalan/akhiran (mis. `infra_Certificate`, `Certificate_xyz`) **tidak dihitung** dan diproses sebagai tag biasa.
2. Resolution notes punya tags (tanpa certificate) → pakai tags itu + logic biasa.
3. Resolution notes **tidak punya tags** → cari kata `Certificate` **utuh** di Short description & Resolution notes. Ketemu → tags = `Certificate`. Kata yang menempel dengan `_` atau `-` (mis. `infra_certificate`) tidak dihitung.

**Logic lama Tag 1 & Tag 2** (sama seperti v2/v3)
- Tag 1 prioritas: `ticket_duplicated` → `cm_bug` → `cm_`
- Tag 2 prioritas: `attachment` → `on-call` → `FEEDBACK_COMPLIMENT` → `user_request` → `user_`
- Ada prefix Tag 1 → Tag 1 = yang terkuat; Tag 2 = prefix Tag 2 terkuat, kalau tidak ada = tag lain paling awal.
- **Tidak ada** prefix Tag 1 dan ada **≥ 2** prefix Tag 2 → Tag 2 = terkuat ke-1, Tag 1 = terkuat ke-2.
- Tidak ada prefix Tag 1 dan hanya 1 prefix Tag 2 → Tag 2 = itu, Tag 1 = tag lain paling awal.
- Tidak ada prefix sama sekali → Tag 1 & Tag 2 = dua tag pertama sesuai urutan.
- Hanya 1 tag → masuk Tag 1, Tag 2 kosong.

## 1. Pengaturan (ubah di sini saja)

In [ ]:
# ---------- file ----------
INPUT_FILE  = "input.xlsx"          # bisa .xlsx / .xls / .csv
SHEET_NAME  = 0                     # khusus Excel: 0 = sheet pertama, atau nama sheet
OUTPUT_FILE = "output_tags.xlsx"

# ---------- nama kolom input ----------
RESOLUTION_COL = "Resolution notes"
SHORT_DESC_COL = "Short description"

# ---------- nama kolom output ----------
TAGS_RAW_COL  = "Tags Raw"            # gabungan tag semua blok, '#' dipertahankan (untuk cek)
TAGS_COL      = "Tags Extracted"      # gabungan tag semua blok, tanpa '#'
NO_RISK_COL   = "remove_Is_High_Risk"
SOURCE_COL    = "Tags Source"
TAG1_COL      = "Tag 1"
TAG2_COL      = "Tag 2"
MODE_COL      = "Tag 1/2 Mode"        # aturan mana yang dipakai untuk Tag 1 & Tag 2

# ---------- Tag 1 & Tag 2 ----------
HASH_PRIORITY        = True         # False -> abaikan '#', Tag 1/2 dari semua tag pakai logic lama
TAG12_SKIP_HIGH_RISK = False        # True  -> "IS High Risk" tidak boleh jadi Tag 1/Tag 2

# ---------- prefix (urut dari yang paling kuat) ----------
TAG1_PREFIXES = ["ticket_duplicated", "cm_bug", "cm_"]
TAG2_PREFIXES = ["attachment", "on-call", "FEEDBACK_COMPLIMENT", "user_request", "user_"]

# ---------- certificate & IS High Risk ----------
CERT_WORDS   = ["certificate", "certificates"]   # tag harus PERSIS salah satu ini (tanpa awalan/akhiran)
CERT_TAG     = "Certificate"        # isi tags kalau certificate terdeteksi
HIGH_RISK    = "IS High Risk"

## 2. Fungsi: ambil tags dari Resolution notes (semua blok, unik)

In [ ]:
import re
from pathlib import Path
import pandas as pd

BR_RE   = re.compile(r"<\s*br\s*/?\s*>", re.I)
HTML_RE = re.compile(r"</?\s*(?:p|div|span|li|ul|ol|b|i|u|strong|em)\s*/?\s*>", re.I)
TAGS_RE = re.compile(r"tags\s*:", re.I)
CTRL_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")


def clean_text(value):
    """<br/> jadi baris baru, buang HTML ringan & karakter aneh."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    s = BR_RE.sub("\n", str(value))
    s = HTML_RE.sub("\n", s)
    s = s.replace("&nbsp;", " ").replace("\r\n", "\n").replace("\r", "\n")
    return CTRL_RE.sub("", s)


def parse_block(block):
    """Satu blok setelah 'Tags:' -> list (tag, pakai_hash). Berhenti di baris yang mengandung ':'."""
    items = []
    for line in block.split("\n"):
        line = line.strip()
        if not line:
            continue
        if ":" in line:
            break
        for item in line.split(","):
            raw = item.strip()
            tag = raw.lstrip("#").strip()
            if tag:
                items.append((tag, raw.startswith("#")))
    return items


def extract_tag_items(text):
    """
    Gabungkan tag dari SEMUA blok 'Tags:' lalu dibuat unik (urutan kemunculan pertama).
    Tag yang sama muncul dengan & tanpa '#' -> dianggap pakai '#'.
    Hasil: list (tag, pakai_hash).
    """
    merged = {}                                   # key casefold -> [tag, pakai_hash]
    for block in TAGS_RE.split(clean_text(text))[1:]:
        for tag, has_hash in parse_block(block):
            key = tag.casefold()
            if key in merged:
                merged[key][1] = merged[key][1] or has_hash
            else:
                merged[key] = [tag, has_hash]
    return [(tag, has_hash) for tag, has_hash in merged.values()]

## 3. Fungsi: logic certificate

In [ ]:
def is_cert_tag(tag):
    """True hanya kalau tag PERSIS 'Certificate'/'Certificates'. 'infra_Certificate' -> False."""
    return tag.strip().casefold() in {w.casefold() for w in CERT_WORDS}


def has_cert_word(text):
    """Cari kata certificate UTUH di teks. 'SSL Certificate renewal' -> True, 'infra_certificate' -> False."""
    words = "|".join(re.escape(w) for w in sorted(CERT_WORDS, key=len, reverse=True))
    return re.search(rf"(?<![\w-])(?:{words})(?![\w-])", text, re.I) is not None


def resolve_tags(resolution_note, short_desc):
    """Kembalikan (items final [(tag, pakai_hash)], tags raw untuk cek, sumber tags)."""
    items = extract_tag_items(resolution_note)
    raw   = ", ".join(("#" if h else "") + t for t, h in items)

    # 1 & 2: Resolution notes punya tags
    if items:
        if any(is_cert_tag(t) for t, _ in items):
            return [(CERT_TAG, False)], raw, "certificate (dari tags)"
        return items, raw, "resolution notes"

    # 3: tidak punya tags -> cari kata certificate di short description & resolution notes
    text = clean_text(short_desc) + " " + clean_text(resolution_note)
    if has_cert_word(text):
        return [(CERT_TAG, False)], raw, "certificate (dari teks)"

    return [], raw, "tidak ada tags"

## 4. Fungsi: Tag 1 & Tag 2 (logic lama + prioritas `#`)

In [ ]:
def prefix_rank(tag, prefixes):
    """Posisi prefix yang cocok (0 = terkuat). None kalau tidak cocok."""
    t = tag.casefold()
    for rank, p in enumerate(prefixes):
        if t.startswith(p.casefold()):
            return rank
    return None


def strongest(tags, prefixes):
    """Index tag yang cocok prefix, diurutkan dari yang terkuat (lalu urutan muncul)."""
    found = [(prefix_rank(t, prefixes), i) for i, t in enumerate(tags)]
    return [i for rank, i in sorted(f for f in found if f[0] is not None)]


def pick_tag1_tag2(tags):
    """LOGIC LAMA (v2/v3) — tidak diubah."""
    n = len(tags)
    if n == 0:
        return "", ""
    if n == 1:
        return tags[0], ""

    t1 = strongest(tags, TAG1_PREFIXES)
    t2 = strongest(tags, TAG2_PREFIXES)
    first_other = lambda skip: next(i for i in range(n) if i != skip)

    if t1:                          # ada prefix Tag 1
        i1 = t1[0]
        i2 = t2[0] if t2 else first_other(i1)
    elif len(t2) >= 2:              # tidak ada Tag 1, ada >= 2 prefix Tag 2
        i2, i1 = t2[0], t2[1]
    elif len(t2) == 1:              # tidak ada Tag 1, cuma 1 prefix Tag 2
        i2 = t2[0]
        i1 = first_other(i2)
    else:                           # tidak ada prefix sama sekali
        i1, i2 = 0, 1

    return tags[i1], tags[i2]


def pick_with_locked(locked, others):
    """
    Hanya 1 tag ber-# (locked): tag ini WAJIB jadi Tag 1 atau Tag 2.
    Slotnya ikut prefix-nya; slot lain diisi dari tag tanpa # (others) pakai logic lama.
    """
    o1 = strongest(others, TAG1_PREFIXES)
    o2 = strongest(others, TAG2_PREFIXES)

    if prefix_rank(locked, TAG1_PREFIXES) is not None:      # tag ber-# tipe Tag 1
        return locked, others[o2[0] if o2 else 0]

    if prefix_rank(locked, TAG2_PREFIXES) is not None:      # tag ber-# tipe Tag 2
        partner = o1[0] if o1 else (o2[0] if o2 else 0)
        return others[partner], locked

    # tag ber-# tanpa prefix
    if o1:
        return others[o1[0]], locked
    if o2:
        return locked, others[o2[0]]
    return locked, others[0]


def pick_tags_v4(items):
    """items: list (tag, pakai_hash) -> (Tag 1, Tag 2, mode)."""
    if TAG12_SKIP_HIGH_RISK:
        items = [(t, h) for t, h in items if t.casefold() != HIGH_RISK.casefold()]
    if not items:
        return "", "", ""

    tags   = [t for t, _ in items]
    hashed = [t for t, h in items if h]
    plain  = [t for t, h in items if not h]

    if not HASH_PRIORITY or not hashed or not plain:   # semua ber-# / semua tanpa #
        return (*pick_tag1_tag2(tags), "semua tag (logic lama)")
    if len(hashed) >= 2:                                # cukup dari tag ber-#
        return (*pick_tag1_tag2(hashed), "hanya tag ber-#")
    return (*pick_with_locked(hashed[0], plain), "1 tag ber-# + tag tanpa #")

## 5. Cek cepat logic (tidak butuh file input)

Pakai pengaturan default di bagian 1. Kalau pengaturan diubah, sebagian test bisa ikut berubah hasilnya.

In [ ]:
# (resolution notes, short description, Tags Extracted, Tag 1, Tag 2)
CASES = [
    # --- semua ber-# / semua tanpa # -> logic lama ---
    ("Tags:\nTags:\n#User_Request\n", "",
     "User_Request", "User_Request", ""),
    ("Val version is aligned with PRD<br/>Tags:<br/>#Infra<br/>Tracker ID:\nTags:", "",
     "Infra", "Infra", ""),
    ("Tags:\n#CM_item, #User_request\nTags:\n#Attachment_missing, #CM_item", "",
     "CM_item, User_request, Attachment_missing", "CM_item", "Attachment_missing"),
    ("Tags:\nUser_request, hyte", "",
     "User_request, hyte", "hyte", "User_request"),
    ("Tags:\nInfra, #Bug\nTags:\n#Infra", "",                        # Infra dihitung ber-#
     "Infra, Bug", "Infra", "Bug"),

    # --- >= 2 tag ber-# -> hanya dari tag ber-# ---
    ("Tags:<br/>Tags:\nTags:\nIS High Risk, #DTS_Transmission_Failing, #CM_BUG_MessageDuplicated_31/32\n"
     "tracker id: 100312 - TRACKER INEXTRACK PMI - Message duplicated [IS High Risk]", "",
     "IS High Risk, DTS_Transmission_Failing, CM_BUG_MessageDuplicated_31/32",
     "CM_BUG_MessageDuplicated_31/32", "DTS_Transmission_Failing"),
    ("Tags:\n#Infra, #Bug, CM_BUG_x", "",                            # CM_BUG_x tanpa # diabaikan
     "Infra, Bug, CM_BUG_x", "Infra", "Bug"),

    # --- cuma 1 tag ber-# -> wajib masuk, slot lain dari tag tanpa # ---
    ("Tags:<br/>#CM_BUG_MessageDuplicated, IS High Risk\nTags:\n#CM_BUG_MessageDuplicated, IS High Risk\n"
     "tracker id: 112952 - TRACKER INEXTRACK PMI - Duplicated submission [IS High Risk]\n", "",
     "CM_BUG_MessageDuplicated, IS High Risk", "CM_BUG_MessageDuplicated", "IS High Risk"),
    ("Tags:\n#User_Request, IS High Risk", "",                      # ber-# tipe Tag 2
     "User_Request, IS High Risk", "IS High Risk", "User_Request"),
    ("Tags:\n#CM_item, User_request, CM_BUG_x", "",                 # CM_item ber-# menang atas CM_BUG_x
     "CM_item, User_request, CM_BUG_x", "CM_item", "User_request"),
    ("Tags:\n#Infra, CM_BUG_x, User_request", "",                   # ber-# tanpa prefix, ada prefix Tag 1
     "Infra, CM_BUG_x, User_request", "CM_BUG_x", "Infra"),
    ("Tags:\n#Infra, User_request, hyte", "",                       # ber-# tanpa prefix, ada prefix Tag 2
     "Infra, User_request, hyte", "Infra", "User_request"),
    ("Tags:\n#Infra, hyte, IS High Risk", "",                       # tidak ada prefix sama sekali
     "Infra, hyte, IS High Risk", "Infra", "hyte"),

    # --- certificate: hanya tag yang PERSIS 'Certificate', ber-# maupun tidak ---
    ("Tags:\n#Certificate, #CM_x", "",
     "Certificate", "Certificate", ""),
    ("Tags:\n#CM_BUG_x, Certificate", "",
     "Certificate", "Certificate", ""),
    ("Tags:\n#Certificates_expired, #CM_x", "",                      # ada akhiran -> tag biasa
     "Certificates_expired, CM_x", "CM_x", "Certificates_expired"),
    ("Tags:\n#infra_Certificate, #User_request", "",                 # ada awalan -> tag biasa
     "infra_Certificate, User_request", "infra_Certificate", "User_request"),
    ("renewed the cert", "SSL Certificate renewal",
     "Certificate", "Certificate", ""),
    ("fix infra_certificate config", "update Certificate_store path",
     "", "", ""),
]


def run_selftest():
    failed = 0
    for i, (res, short, exp_tags, exp1, exp2) in enumerate(CASES, 1):
        items, raw, src = resolve_tags(res, short)
        t1, t2, mode = pick_tags_v4(items)
        got = (", ".join(t for t, _ in items), t1, t2)
        expected = (exp_tags, exp1, exp2)
        ok = got == expected
        failed += not ok
        print(f"[{'OK ' if ok else 'FAIL'}] case {i:>2} | raw: {raw!r}")
        print(f"            Tag 1: {t1!r} | Tag 2: {t2!r} | mode: {mode or '-'} | sumber: {src}")
        if not ok:
            print(f"        got     : {got}")
            print(f"        expected: {expected}")
    print("\n" + ("SEMUA TEST LULUS" if not failed else f"{failed} test GAGAL"))


run_selftest()

## 6. Baca file input

In [ ]:
def find_col(df, name):
    """Cari nama kolom tanpa peduli huruf besar/kecil & spasi."""
    lookup = {str(c).strip().casefold(): c for c in df.columns}
    return lookup.get(name.strip().casefold())


if Path(INPUT_FILE).suffix.lower() == ".csv":
    df = pd.read_csv(INPUT_FILE, dtype=object, sep=None, engine="python", encoding="utf-8-sig")
else:
    df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, dtype=object)

res_col   = find_col(df, RESOLUTION_COL)
short_col = find_col(df, SHORT_DESC_COL)

if res_col is None:
    raise KeyError(f"Kolom '{RESOLUTION_COL}' tidak ada. Kolom tersedia: {list(df.columns)}")
if short_col is None:
    print(f"Peringatan: kolom '{SHORT_DESC_COL}' tidak ada, pencarian certificate hanya di Resolution notes.")

print("Jumlah baris :", len(df))
print("Kolom        :", list(df.columns))

## 7. Jalankan logic

In [ ]:
short_values = df[short_col] if short_col is not None else [""] * len(df)
resolved = [resolve_tags(r, s) for r, s in zip(df[res_col], short_values)]
picked   = [pick_tags_v4(items) for items, _, _ in resolved]

df[TAGS_RAW_COL] = [raw for _, raw, _ in resolved]
df[TAGS_COL]     = [", ".join(t for t, _ in items) for items, _, _ in resolved]
df[SOURCE_COL]   = [src for _, _, src in resolved]
df[NO_RISK_COL]  = [
    ", ".join(t for t, _ in items if t.casefold() != HIGH_RISK.casefold())
    for items, _, _ in resolved
]
df[TAG1_COL] = [p[0] for p in picked]
df[TAG2_COL] = [p[1] for p in picked]
df[MODE_COL] = [p[2] for p in picked]

print("Selesai:", len(df), "baris diproses\n")
print(df[SOURCE_COL].value_counts().to_string(), "\n")
print(df[MODE_COL].replace("", "(tidak ada tag)").value_counts().to_string())

## 8. Lihat hasil & simpan ke Excel

In [ ]:
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 240)

print(df[[TAGS_RAW_COL, TAGS_COL, SOURCE_COL, TAG1_COL, TAG2_COL, MODE_COL]].to_string())

df.to_excel(OUTPUT_FILE, index=False)
print(f"\nTersimpan: {OUTPUT_FILE}")